In [ ]:
"""Seed-1 replication scheduled before seed-0 outcomes, preserving all five folds.

The builder appends a main call with the frozen payload and review manifest.
No dataset upload, network request, package installation or production prediction.
"""
from __future__ import annotations

import base64
from datetime import datetime, timezone
import gzip
import hashlib
import importlib.metadata
import io
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import tarfile
import traceback


def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b""):
            h.update(block)
    return h.hexdigest()


def utc():
    return datetime.now(timezone.utc).isoformat()


def save(path, value):
    with path.open("x", encoding="utf-8") as stream:
        json.dump(value, stream, indent=2, default=str)
        stream.write("\n")


def command_run(command, log_path, *, cwd, env):
    with log_path.open("x", encoding="utf-8") as log:
        with subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as child:
            for line in child.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            return child.wait()


def main(payload64, review):
    os.environ.update({"CUDA_VISIBLE_DEVICES": "0", "OMP_NUM_THREADS": "2",
                       "OPENBLAS_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
                       "PYTHONHASHSEED": "0", "PYTHONUNBUFFERED": "1"})
    work = Path("/kaggle/working/neural_seed1_r1")
    work.mkdir()
    save(work / "runtime_review.json", review)
    save(work / "started.json", {"utc": utc(), "status": "preflight"})
    payload = base64.b64decode(payload64, validate=True)
    if hashlib.sha256(payload).hexdigest() != review["payload_sha256"]:
        raise ValueError("Frozen code payload hash mismatch")
    code = work / "code"
    expected = {item["path"]: item for item in review["code_allowlist"]}
    with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
        members = archive.getmembers()
        if {m.name for m in members} != set(expected) or len(members) != len(expected):
            raise ValueError("Payload differs from the reviewed source allowlist")
        for member in members:
            target = (code / member.name).resolve()
            if not member.isfile() or not target.is_relative_to(code.resolve()):
                raise ValueError("Non-regular or escaping code archive entry")
            content = archive.extractfile(member).read()
            if hashlib.sha256(content).hexdigest() != expected[member.name]["sha256"]:
                raise ValueError(f"Code hash mismatch: {member.name}")
            target.parent.mkdir(parents=True, exist_ok=True)
            with target.open("xb") as stream:
                stream.write(content)
    dataset = Path("/kaggle/input/vcc-rete-contesti-r2")
    if not dataset.is_dir():
        candidates = [p for p in Path("/kaggle/input").rglob("manifest.json")
                      if "vcc-rete-contesti-r2" in p.parts]
        if len(candidates) != 1:
            raise FileNotFoundError("Cannot uniquely locate the reviewed existing dataset")
        dataset = candidates[0].parent
    if sha256(dataset / "manifest.json") != review["dataset_manifest_sha256"]:
        raise ValueError("Dataset manifest differs from the reviewed r2 snapshot")
    if {p.name for p in dataset.iterdir() if p.is_file()} != set(review["input_files"]):
        raise ValueError("Input file list differs from the reviewed r2 inventory")
    for name, expected_input in review["input_files"].items():
        path = dataset / name
        if not path.is_file() or path.stat().st_size != expected_input["bytes"]:
            raise ValueError(f"Reviewed input size differs: {name}")
        if expected_input["sha256"] is not None and sha256(path) != expected_input["sha256"]:
            raise ValueError(f"Reviewed input hash differs: {name}")
    data_manifest = json.loads((dataset / "manifest.json").read_text())
    if data_manifest["format"] != "rete_contesti/1":
        raise ValueError("Unexpected dataset format")
    import numpy as np
    import pandas as pd
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("This reviewed run requires a free Kaggle CUDA GPU; CPU fallback is disabled")
    environment = {"utc": utc(), "python": sys.version, "platform": platform.platform(),
                   "cuda_visible_devices": os.environ["CUDA_VISIBLE_DEVICES"],
                   "torch_cuda": torch.version.cuda, "cuda_device_count_visible": torch.cuda.device_count(),
                   "gpu_name": torch.cuda.get_device_name(0), "versions": {}}
    for package in ("torch", "numpy", "pandas", "scipy", "kaggle"):
        try:
            environment["versions"][package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            environment["versions"][package] = None
    save(work / "environment.json", environment)
    print(json.dumps(environment, indent=2), flush=True)
    files = []
    for name, expected_bytes in data_manifest["bytes"].items():
        path = dataset / name
        actual_bytes = path.stat().st_size
        if actual_bytes != expected_bytes:
            raise ValueError(f"Dataset file size mismatch: {name}")
        entry = {"name": name, "bytes": actual_bytes,
                 "sha256": sha256(path) if actual_bytes < 30_000_000 else None}
        if name.endswith(".npy"):
            array = np.load(path, mmap_mode="r", allow_pickle=False)
            entry.update({"shape": list(array.shape), "dtype": str(array.dtype),
                          "mmap": isinstance(array, np.memmap), "writeable": bool(array.flags.writeable)})
            if not entry["mmap"] or entry["writeable"]:
                raise ValueError(f"Read-only mmap requirement failed: {name}")
            if name in {"raw.npy", "se.npy", "shrunk.npy"}:
                if list(array.shape) != [data_manifest["counts"]["rows"], data_manifest["counts"]["genes"]]:
                    raise ValueError(f"Effect array dimensions differ: {name}")
            del array
        files.append(entry)
    save(work / "input_files.json", {"dataset": review["dataset"], "counts": data_manifest["counts"],
                                    "manifest_sha256": review["dataset_manifest_sha256"], "files": files,
                                    "large_array_identity": "immutable Kaggle dataset version, manifest, file sizes, shapes and dtypes"})
    report = code / "reports/analisi/lead_scientist_2026-09-29"
    families = review["fold_order"]
    outcomes = []
    for family in families:
        out = work / "folds" / f"C_{family}_s1"
        command = [sys.executable, str(report / "train_neural_sources.py"), "--data", str(dataset),
                   "--out", str(out), "--holdout", family, "--regime", "C", "--device", "cuda",
                   "--seed", "1", "--selection-seed", "20260929", "--steps", "1000",
                   "--eval-every", "50", "--patience", "5", "--batch", "16", "--gene-batch", "1024",
                   "--validation-genes", "2048", "--test-targets", "512", "--validation-targets", "128", "--lr", "0.001"]
        start = utc()
        save(work / f"start_{family}.json", {"started_utc": start, "family": family, "argv": command})
        try:
            rc = command_run(command, work / f"console_{family}.log", cwd=code, env=os.environ.copy())
            result = {"family": family, "returncode": rc, "started_utc": start, "finished_utc": utc(),
                      "out": str(out), "complete": rc == 0 and (out / "summary.json").is_file()
                                   and (out / "per_target.csv").is_file()}
        except Exception:
            result = {"family": family, "returncode": None, "started_utc": start, "finished_utc": utc(),
                      "out": str(out), "complete": False, "exception": traceback.format_exc()}
        outcomes.append(result)
        save(work / f"status_{family}.json", result)
        print(json.dumps(result), flush=True)
    all_five = len(outcomes) == 5 and all(item["complete"] for item in outcomes)
    readout_rc = None
    if all_five:
        command = [sys.executable, str(report / "read_neural_verified.py"), "--runs",
                   *[item["out"] for item in outcomes], "--out", str(work / "cross_family")]
        readout_rc = command_run(command, work / "cross_family.log", cwd=code, env=os.environ.copy())
    save(work / "completion.json", {"finished_utc": utc(), "folds": outcomes,
                                    "all_five_completed": all_five, "cross_family_returncode": readout_rc,
                                    "status": "complete" if all_five and readout_rc == 0 else "incomplete",
                                    "not_vcc_score": True, "submission_authorized_by_this_run": False})
    if not all_five or readout_rc != 0:
        raise RuntimeError("All five folds were attempted; inspect preserved status/log files. No partial-family verdict.")
    # Additive post-training sensitivity. Never changes completion, promotion or
    # scientific thresholds; preserve an explicit failure separately if it fails.
    diagnostic = {"started_utc": utc(), "role": "additive sensitivity, not a promotion criterion"}
    try:
        command = [sys.executable, str(report / "neural_external_validation/cluster_pds.py"), "--runs",
                   *[item["out"] for item in outcomes], "--data", str(dataset),
                   "--out", str(work / "cluster_diagnostic")]
        rc = command_run(command, work / "cluster_diagnostic.log", cwd=code, env=os.environ.copy())
        diagnostic.update({"returncode": rc, "complete": rc == 0,
                           "changes_primary_verdict": False})
    except Exception:
        diagnostic.update({"returncode": None, "complete": False, "exception": traceback.format_exc(),
                           "changes_primary_verdict": False})
    diagnostic["finished_utc"] = utc()
    save(work / "cluster_diagnostic_status.json", diagnostic)


main('...', {'prepared_utc': '2026-09-29T19:22:17.442679+00:00', 'kernel': 'davidmaisterx/vcc-lead-neural-seed1-r1', 'private': True, 'dataset': 'davidmaisterx/vcc-rete-contesti-r2', 'dataset_id': 12242239, 'dataset_manifest_sha256': '2e59376b103dc6578b9a52e3c37c655207567ecc0375481bc5e3560e7fb8b32a', 'original_seed0_payload_sha256': '283d4f592f87fcd969a4dec80d2138ce1568eeef1ac2c4dd0485e641185780bf', 'payload_sha256': '4c5b7b53069b0642d0b9273c87c1128b6ccf299275e816271e3f79b7cbd52665', 'payload_bytes': 40286, 'code_allowlist': [{'path': 'reports/analisi/lead_scientist_2026-09-29/EMENDAMENTO_LETTORE_NEURALE_01.md', 'bytes': 1679, 'sha256': 'eb318d4e79a73ade2d72e489be2e7a65bc37ee0f9210f71aa500a3d0497bec70'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/EMENDAMENTO_NEURALE_SCHEDULING_01.md', 'bytes': 2647, 'sha256': '9da138fe769fa0919fb087d6f1e83910e8ac1831816115aa93e5dbf401b53c9d'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/PROTOCOLLO_NEURALE.md', 'bytes': 7452, 'sha256': '7febed5942476f38f98af8969d7cc38f22153e419c90014ea5b24f3c05068374'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/neural_external_validation/PROTOCOLLO_DIAGNOSTICA.md', 'bytes': 5916, 'sha256': '5b4242c69aa2a82ee67e2cb44aad0c92e9dcad69c277dcc10e47875dd971c0c5'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/neural_external_validation/cluster_pds.py', 'bytes': 9235, 'sha256': '4413ad2662b6315c9f08c396a766ba6d9d4fa066d22ad5661b06226eb416c1e9'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/neural_external_validation/test_cluster_pds.py', 'bytes': 3669, 'sha256': 'ab3273f9c6498102bd472432b4d5ba464e7915b10ed9f572c284fe5bd462a493'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/neural_reader/r1/read_neural_sources.py', 'bytes': 3626, 'sha256': '95f0f37610778d26b09806cfed5614d054c8f6fa181a6844480f5277f47c8f22'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/neural_seed1_runner_r1.py', 'bytes': 10301, 'sha256': 'a20991d8d1c657807ece97690dc4517448ea80b949cb9608d3f9d175b27cd902'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/neural_sources.py', 'bytes': 14657, 'sha256': 'fdfc6e7cadc153e161d9679fe0ee3c3ca6a846e71df85b328ae9a726133f55b1'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/read_neural_sources.py', 'bytes': 3649, 'sha256': '21b936b387ec734e0aaeb4b85f982f127a465a84fb3a77dc89917190881a8243'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/read_neural_verified.py', 'bytes': 1735, 'sha256': '803bb35a72b7f524064b6c30c943d47c211143f12164fe2652c20362e3c25c48'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/train_neural_sources.py', 'bytes': 19483, 'sha256': '2b4e9734a0ea630010e0b83c8cbbb0c48fd1707e5356c3834ee5dbb954d55f72'}, {'path': 'reports/analisi/lead_scientist_2026-09-29/verify_neural_runs.py', 'bytes': 5185, 'sha256': 'e90e095c25f7e4e17c0cc0e4cfad71f87b57eababf49765db9c5ec624ff0e3b4'}, {'path': 'reports/modelli/rete_contesti_2026-09-27/pool.py', 'bytes': 45527, 'sha256': '10bd001536c013be8937c0ea321e0e9b7d16416ac525104f6fe0ef678f0940ce'}], 'input_files': {'axis.csv': {'bytes': 137824, 'sha256': 'da3090db3d1b27b62ab513f4ecc93a5bf1ef2e6ac9c11e825d5dad2f19c03ccb'}, 'basal.npy': {'bytes': 1186240, 'sha256': 'e1e2c33f8e31db2bef295f2d1192597a314776fddd94b6660548627403e0fd9e'}, 'cis_index.npy': {'bytes': 13492, 'sha256': 'eeb844a6a7acbe33adcf239efbb2c3d0ab5a62ca9b52a4d6ae8dc30a60fd9b58'}, 'cis_indptr.npy': {'bytes': 154256, 'sha256': 'd232935fd8bab0b90ef74e86635926db23363c6234c409ceaa49d902b9b8cccb'}, 'contexts.csv': {'bytes': 1457, 'sha256': '2847d53b3e35308e1e497ba6d700032c0066934733fd2a83873848db78ef4fd6'}, 'genes.csv': {'bytes': 194675, 'sha256': '4009e0632f3d9854c0029b5e56cb21a23b821dba4fb430a1cafc18a348d62225'}, 'manifest.json': {'bytes': 11508, 'sha256': '2e59376b103dc6578b9a52e3c37c655207567ecc0375481bc5e3560e7fb8b32a'}, 'panel_targets.txt': {'bytes': 2194, 'sha256': 'dc47a19604b25553c2c0c4f6d2fcf1592671c1658f89be01fce0219138413cd4'}, 'partner_index.npy': {'bytes': 447852, 'sha256': 'b8199d79e37bf9549f74043637df6a3d2144597717e5165619505dbd4d2d6813'}, 'partner_indptr.npy': {'bytes': 154256, 'sha256': 'd89cd8428990bf066a27c81b6ea47cb26942c8f24ffa49af70f924666beb347e'}, 'partner_score.npy': {'bytes': 447852, 'sha256': 'b1855fa650bae179997f72b30c865b118bcbdf73fa083bed42ab05cbbe3d554a'}, 'raw.npy': {'bytes': 2903590784, 'sha256': None}, 'row_context.npy': {'bytes': 438472, 'sha256': 'd5a55bc818cde00e1dd4aad216904a7f1a80b43543aef21c834c1a531b5e0fab'}, 'row_ncells.npy': {'bytes': 438472, 'sha256': 'eceede9c26fb87be59fc3778a430f3be9fbf913382f42c84df4efcd0fce6a447'}, 'row_own.npy': {'bytes': 876816, 'sha256': '6266cc54604ae895dbfe3edaab7117372e2131cc434cfd94b2a6a4a39ee84dbe'}, 'row_target.npy': {'bytes': 438472, 'sha256': 'c9e27f485f22409baa5b287ee6cef45a51cfb10bc1adecb76ec51aaff7fe8698'}, 'se.npy': {'bytes': 2903590784, 'sha256': None}, 'shrunk.npy': {'bytes': 2903590784, 'sha256': None}, 'targets.csv': {'bytes': 2279047, 'sha256': '3da48d5558b230a11b7950d41d3081794f78b12302071fd9de122e7755ba817c'}}, 'fold_order': ['k562', 'cd4', 'orion', 'ipsc', 'rpe1'], 'regime': 'C', 'seed': 1, 'selection_seed': 20260929, 'same_original_training_code': True, 'schedule_before_seed0_outcome': True, 'requires_positive_seed0_for_scheduling': False, 'read_all_folds_even_if_seed0_fails': True, 'no_seed_cherry_picking': True, 'production_still_requires_verified_positive_gate': True, 'accelerator_requested': 'NvidiaTeslaT4', 'cuda_visible_devices': '0', 'quota_observed_hours_remaining': 7.59, 'concurrent_slot_availability': None, 'new_data_uploaded': False, 'network_or_package_install_in_runtime': False, 'no_production_fit': True, 'cross_family_requires_all_five': True, 'cluster_changes_primary_verdict': False})
